In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re

url = "https://ldcom365.sharepoint.com"


history=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx')

ref_date = datetime.today() - timedelta(days=5)

ref_year = ref_date.year
ref_month = ref_date.month



In [0]:
if ref_month == 12:
    next_month = 1
    next_year = ref_year + 1
else:
    next_month = ref_month + 1
    next_year = ref_year

# Now filter
filtered_df = history[
    ((history['Date'].dt.year == ref_year) & (history['Date'].dt.month == ref_month)) |
    ((history['Date'].dt.year == next_year) & (history['Date'].dt.month == next_month))
].copy()


filtered_df_SBM = filtered_df[filtered_df['Product'] == 'SBM'].copy()
filtered_df_SBM=filtered_df_SBM[filtered_df_SBM['Origin']=='ARG']
filtered_df_SBM['Date'] = filtered_df_SBM['Date'].dt.date


# Step 4: Adjust dates where day >= 28, but only if Status == 'Announced'
def adjust_date(row, ref_year, ref_month):
    date = row['Date']
    status = row['Status']
    
    if status == 'Announced' and date.year == ref_year and date.month == ref_month and date.day >= 27:
        # If December, move to Jan 5th next year
        if ref_month == 12:
            return pd.Timestamp(year=ref_year + 1, month=1, day=5)
        else:
            return pd.Timestamp(year=ref_year, month=ref_month + 1, day=5)
    return date

# Apply to each row (need axis=1 because we work with multiple columns)
filtered_df_SBM['Date'] = filtered_df_SBM.apply(lambda row: adjust_date(row, ref_year, ref_month), axis=1)

filtered_df_SBM['Date'] = pd.to_datetime(filtered_df_SBM['Date'])

filtered_df_SBM['Month']=filtered_df_SBM['Date'].dt.month
filtered_df_SBM['Year']=filtered_df_SBM['Date'].dt.year


# Ensure 'Flag' column exists
filtered_df_SBM['Flag'] = None

# Build the mask: day between 25 and 27 AND Status is 'Announced'
mask_review = (
    filtered_df_SBM['Date'].apply(lambda d: 25 <= d.day <= 26) &
    (filtered_df_SBM['Status'] == 'Announced')
)

# Apply the flag
filtered_df_SBM.loc[mask_review, 'Flag'] = 'TO BE REVIEWED'




In [0]:
# Step 1: Compute next_month and next_year
if ref_month == 12:
    next_month = 1
    next_year = ref_year + 1
else:
    next_month = ref_month + 1
    next_year = ref_year

# Step 2: Work directly on history
# Build a mask for SBM and ARG
mask_sbm_arg = (history['Product'] == 'SBM') & (history['Origin'] == 'ARG')

# Step 3: Adjust dates where day >= 28, Status == 'Announced', and month == ref_month
def adjust_date(row):
    date = row['Date']
    status = row['Status']
    
    if (status == 'ANNOUNCED') and (date.year == ref_year) and (date.month == ref_month) and (date.day >= 28):
        if ref_month == 12:
            return pd.Timestamp(year=ref_year + 1, month=1, day=5)
        else:
            return pd.Timestamp(year=ref_year, month=ref_month + 1, day=5)
    return date

# Apply adjustment only on SBM/ARG rows
history.loc[mask_sbm_arg, 'Date'] = history.loc[mask_sbm_arg].apply(adjust_date, axis=1)

# Step 4: Update Year and Month based on new Date for SBM/ARG
history.loc[mask_sbm_arg, 'Month'] = history.loc[mask_sbm_arg, 'Date'].dt.month
history.loc[mask_sbm_arg, 'Year'] = history.loc[mask_sbm_arg, 'Date'].dt.year

# Step 5: Initialize or clear Flag column
if 'Flag' not in history.columns:
    history['Flag'] = None

# Step 6: Build the mask for "TO BE REVIEWED"
mask_review = (
    (history['Product'] == 'SBM') &
    (history['Origin'] == 'ARG') &
    (history['Status'] == 'ANNOUNCED') &
    (history['Date'].dt.day.between(25, 27)) &
    (history['Date'].dt.month == ref_month) &  # Important to still match ref_month
    (history['Date'].dt.year == ref_year)      # and ref_year
)

# Step 7: Apply the flag
history.loc[mask_review, 'Flag'] = 'TO BE REVIEWED'


In [0]:
#sp_mgr.rm('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx')
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx',history,index=False)

In [0]:
body_1 = """
<p>Hi,</p>

<p><strong>Lineups have been added to the history. Please check the rows in the "Flag" column labeled as "TO BE REVIEWED", "Alongside", "At Roads".</strong></p>

<p>If you want to roll the vessel to the next month, write <strong> "Next" <strong> in the "Flag" column; otherwise, leave the cell empty.</p>

<p><strong>The report will be sent Monday at 08:00. Vessels not reviewed by then will remain as they are.<strong></p>


<p>Path: https://ldcom365.sharepoint.com/:f:/s/GRP-TradingLineups/EjtJnlE871BHqsJwVYGJQtABv_Wuo6fEaSwJddYLecrRLQ?e=kqQHNj </p>

<p>Dashboard:  https://app.powerbi.com/links/5cMlqugPbP?ctid=40a64d0b-f2f9-4a34-b1b3-0992ac0e5e4e&pbi_source=linkShare  </p>


"""

meal=["florian.girardi-ext@ldc.com","michelle.lambrechts@ldc.com","SANTIAGO.CARBAJAL@ldc.com"]

moi=["florian.girardi-ext@ldc.com"]

LDCDataAccessLayerPy.mail.mail_send(to=moi, subject=f'ARG Lineups SBM {datetime.now().strftime("%d-%m")}',from_addr="florian.girardi-ext@ldc.com",body=body_1,mime_type="html")

### OSA

In [0]:
country_mapping={"S. ARABIA":"SAUDI ARABIA","SAUDI ARABI":"S. ARABIA"}


country_region_mapping = {
    "ALGERIA": "AFRICA",
    'ALEGERIA':'AFRICA',
    "ARGENTINA": "SOUTH AMERICA",
    "BANGLADESH": "ASIA",
    "BELGIUM": "EU",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "CHINA": "ASIA",
    "COLOMBIA": "SOUTH AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "CYPRUS": "EU",
    "CYPRUS/GREECE": "EU",
    "DENMARK": "EU",
    "EGYPT": "AFRICA",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "EU-28":"EU",
    "GREECE/ISRAEL": "EU",
    "GREECE/ITALY": "EU",
    "HOLLAND": "EU",
    "HOLLAND/GERMANY": "EU",
    "HONDURAS": "CENTRAL AMERICA",
    "INDONESIA": "SE ASIA",
    "IRAN": "ME ASIA",
    "IRELAND": "EU",
    "ISRAEL": "ME ASIA",
    "ISRAEL/GREECE": "EU",
    "ITALY": "EU",
    "IVORY COAST": "AFRICA",
    "JAPAN": "ASIA",
    "JORDAN": "ME ASIA",
    "KENYA": "AFRICA",
    "KOREA": "ASIA",
    "LEBANON": "ME ASIA",
    "LITHUANIA": "EU",
    "MALAYSIA": "SE ASIA",
    "MAURITIUS": "AFRICA",
    "MEXICO": "CENTRAL AMERICA",
    "MOROCCO": "AFRICA",
    "NETHERLANDS": "EU",
    "NETHERLANDS/GERMANY": "EU",
    "NIGERIA": "AFRICA",
    "PANAMA": "CENTRAL AMERICA",
    "PERU": "SOUTH AMERICA",
    "PHILIPPINES": "SE ASIA",
    "PORTUGAL": "EU",
    "PUERTO RICO": "CENTRAL AMERICA",
    "ROMANIA": "EU",
    "RUSSIA": "ASIA",
    "SAF": "AFRICA",
    "SAUDI ARABIA": "ME ASIA",
    "SENEGAL": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SOUTH KOREA": "ASIA",
    "SPAIN": "EU",
    "SYRIA": "ME ASIA",
    "TAIWAN": "ASIA",
    "TBC": "TBC",
    "THAILAND": "SE ASIA",
    "TUNISIA": "AFRICA",
    "TURKEY": "ME ASIA",
    "U.ARAB EMIRAT": "ME ASIA",
    "UAE": "ME ASIA",
    "UK": "EU",
    "UNITED KINGDOM": "EU",
    "POLAND": "EU",
    "SLOVENIA": "EU",
    "LITUANIA": "EU",
    "CROATIA": "EU",
    "GREECE + CYPRUS":'EU',

    "UNITED ARAB EMIRATES": "ME ASIA",
    "URUGUAY": "SOUTH AMERICA",
    "USA": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "VIETNAM": "SE ASIA",
    "YEMEN": "ME ASIA",
    "": "NOT AVAILABLE",
    "MOZAMBIQUE": "AFRICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "US": "NORTH AMERICA",
    "Z. OTHER EAST AFRICA": "AFRICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "AUSTRALIA": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "OMAN": "ME ASIA",
    "KUWAIT": "ME ASIA",
    "SWITZERLAND": "EU",
    "IRAQ": "ME ASIA",
    "HAITI": "CENTRAL AMERICA",
    "CANADA": "NORTH AMERICA",
    "TRINIDAD": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "ANGOLA": "AFRICA",
    "Z. OTHER FSU": "EUROPE",
    "NORWAY": "EU",
    "NOT AVAILABLE": "",
    "ECUADOR":"SOUTH AMERICA",
    'GUYANA':"SOUTH AMERICA",
    'TANZANIA':'AFRICA',
    'LUANDA':'AFRICA',
    'LIBYA':'AFRICA', 
    'PHILIPPINNES':'SE ASIA',
    'REUNION ISLAND':'AFRICA',
    'GHANA':'AFRICA',
    'CONGO':'AFRICA',
    'NAMIBIA':'AFRICA',
    'CAPE VERDE':'AFRICA',
    'MAURITANA':'AFRICA',
    'UGANDA':'AFRICA',
    'ZIMBABWE':'AFRICA',
    'MALI':'AFRICA',
    'SUDAN':'AFRICA',
    'MAURITANIA':'AFRICA', 
    'ETHIOPIA':'AFRICA',
    'RUANDA':'AFRICA',
    'RWANDA':'AFRICA',
    'BURUNDI':'AFRICA', 
    'LYBIA':'AFRICA',
    'MAURITUS IS':'AFRICA',
    'LATVIA':'EU',
    'ESTONIA':'EU',

    'CAMEROON':'AFRICA',
    'IVORY COST':'AFRICA',
    'REUNION':'AFRICA',
    'DOM. REP':'CENTRAL AMERICA',
    'DOM REP.':'CENTRAL AMERICA',
    'DOM REP;':'CENTRAL AMERICA',
    'DOM. REP.':'CENTRAL AMERICA',
    'DOM. REP;':'CENTRAL AMERICA',
    'DOM REP':'CENTRAL AMERICA',
    'TRINIDAD & TOBAGO':'CENTRAL AMERICA',
    'GEORGIA':'ME ASIA',
    'KUWEIT':'ME ASIA',
    'BAHREIN':'ME ASIA',
    'DJBOUTI':'ME ASIA',
    'SAUDI ARABIA ':'ME ASIA',

    
    'BRUNEI':'SE ASIA',
    'MYANMAR':'SE ASIA',
    'MALAYSIA':'SE ASIA',
    'Malaysia':'SE ASIA',
    'PHILIPINES':'SE ASIA',
    'INDIA':'ASIA',
    'BELARUS':'ASIA',

    'NEW ZELAND':'OCEANIA',
    'MAURITUIS':'AFRICA',
    'U.A.E.':'ME ASIA',
    'EAU':'ME ASIA',
    'LEBANNON':'ME ASIA',
    'HOLANDA':'EU',
    'PAKISTAN':'ASIA',
    'PAKISTAN ':'ASIA',
    'RUSSIAN FEDERATION':'ASIA',
    'UNITED STATES':'ASIA',
    'TURKEY ':'ME ASIA',

    'SOUTH KOREA ':'ASIA',
    'JAPAN  ':'ASIA',
    'MALAYSIA  ':'SE ASIA',
    'IRAK':'ME ASIA', 
    'BRASIL':'SOUTH AMERICA',
    'MARRUECOS':'AFRICA',
    'ECUADOR ':'SOUTH AMERICA',
    'PARAGUAY':'SOUTH AMERICA',
    'BRAZIL ':'SOUTH AMERICA',
    'CHILE ':'SOUTH AMERICA',
    'BOLIVIA':'SOUTH AMERICA',
    'MADAGASCAR':'AFRICA',
    'GABON':'AFRICA',
    'SENEGAL ':'AFRICA',
    'GAMBIA':'AFRICA',
    'QATAR':'ME ASIA', 
    'BAHRAIN':'ME ASIA',
    'LEBANON ' :'ME ASIA',
    'BURKINA FASO':'AFRICA',
    'ALGERIA ':'AFRICA',
    'MALAWI':'AFRICA',
    'GUINEA':'AFRICA',
    'TOGO':'AFRICA',
    'LIBERIA':'AFRICA',
    'DJIBOUTI':'AFRICA',
    'VIETNAM ':'SE ASIA',
    "OTH_AFR":'AFRICA',
    "OTH_AMER":"SOUTH AMERICA",
    "OTH_EME":"ME ASIA",
    "OTH_ASIA":"SE ASIA",
    "S. ARABIA":"ME ASIA",
    "WORLD":"WORLD",
    np.nan:'UNKNOWN'
    }


In [0]:
osa=sp_mgr.read_pd_from_excel('/sites/grp-oilseedssnd/Shared%20Documents/MainFile/Oilseeds_Analytics_version2.xlsm',sheet_name="mtx_sbm")
ARG_SBM = osa.loc[:, (osa.columns[0],) + tuple(osa.columns[1:][osa.iloc[1, 1:].astype(str).str.contains('arg', case=False, na=False)])]
ARG_SBM.columns = ARG_SBM.iloc[0]    # Set first row as header
ARG_SBM = ARG_SBM[1:]                # Drop the first row from the data
ARG_SBM.reset_index(drop=True, inplace=True)  
ARG_SBM = ARG_SBM[2:] 
# Try to convert the first column to datetime
ARG_SBM.iloc[:, 0] = pd.to_datetime(ARG_SBM.iloc[:, 0], errors='coerce')

# Keep only rows where the first column could be parsed as a datetime
ARG_SBM = ARG_SBM[ARG_SBM.iloc[:, 0].notna()]

# Optional: reset index
ARG_SBM.reset_index(drop=True, inplace=True)

ARG_SBM = ARG_SBM.rename(columns={'SBM': 'date'})
ARG_SBM['month']=ARG_SBM['date'].dt.month
ARG_SBM['year']=ARG_SBM['date'].dt.year

current_year = datetime.now().year
# Filter the DataFrame
df_current_year = ARG_SBM[ARG_SBM['year'] == current_year]
# Pivot so that countries are rows and months are columns
pivot_df = df_current_year.set_index('date').drop(columns=['month', 'year']).T
# Set column names to the months
pivot_df.columns = df_current_year['month'].values
pivot_df = pivot_df[~(pivot_df == 0).all(axis=1)]

pivot_df = pivot_df.reset_index().rename(columns={'index': 'Country'})

pivot_df

In [0]:
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/MEAL/Dest_forecasts_meal.xlsx',pivot_df,index=False)